In [1]:
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
from typing import List, Set, Tuple, Optional
import pandas as pd
import numpy as np

# Combine dataframe

In [2]:
def collect_csvs(dir_path: str) -> List[str]:
    return [
        os.path.join(dir_path, f)
        for f in sorted(os.listdir(dir_path))
        if f.lower().endswith(".csv")
    ]


def infer_numeric_cols_from_first_file(
    path: str,
    threshold: float = 0.95,
    label_col: str = "Label"
) -> Set[str]:
    df = pd.read_csv(path, low_memory=False)
    numeric_cols = set()

    for c in df.columns:
        if c == label_col:
            continue

        s = (
            df[c]
            .dropna()
            .astype(str)
            .str.strip()
            .replace(
                {
                    "": np.nan,
                    "Infinity": np.nan,
                    "inf": np.nan,
                    "-inf": np.nan,
                }
            )
        )

        if len(s) == 0:
            continue

        if pd.to_numeric(s, errors="coerce").notna().mean() >= threshold:
            numeric_cols.add(c)

    return numeric_cols

In [3]:
def combine_to_df_for_range(
    input_dir: str,
    file_range: Tuple[int, int],
    mask: int = 10,
    label_col: str = "Label",
    numeric_threshold: float = 0.95
) -> pd.DataFrame:

    start, stop = file_range
    if stop <= start:
        raise ValueError("file_range must satisfy stop > start")

    paths = collect_csvs(input_dir)
    selected = paths[start:stop]

    if not selected:
        raise FileNotFoundError(f"No files in range {file_range}")

    drop_cols = {"Dst Port", "Timestamp"}
    gap = max(0, mask - 1)

    numeric_cols = infer_numeric_cols_from_first_file(
        selected[0],
        threshold=numeric_threshold,
        label_col=label_col
    )

    canonical_columns = None
    parts = []

    for path in selected:
        part = pd.read_csv(path, low_memory=False)
        part = part.drop(
            columns=[c for c in drop_cols if c in part.columns],
            errors="ignore"
        )

        if canonical_columns is None:
            canonical_columns = list(part.columns)
            if label_col not in canonical_columns:
                canonical_columns.append(label_col)

        part = part.reindex(columns=canonical_columns)

        for c in numeric_cols:
            if c in part.columns:
                part[c] = pd.to_numeric(
                    part[c]
                    .astype(str)
                    .replace(
                        {
                            "Infinity": np.nan,
                            "inf": np.nan,
                            "-inf": np.nan,
                        }
                    ),
                    errors="coerce"
                )

        part = part.ffill()

        if gap > 0:
            gap_df = pd.DataFrame(
                0,
                index=range(gap),
                columns=canonical_columns
            )
            gap_df[label_col] = "Benign"
            parts.append(gap_df)

        parts.append(part)

    return pd.concat(parts, ignore_index=True)

In [4]:
combined_train = combine_to_df_for_range(
    input_dir="/kaggle/input/ids-intrusion-csv",
    file_range=(0, 2),
    mask=10,
    label_col="Label",
    numeric_threshold=0.95
)

combined_supplement = combine_to_df_for_range(
    input_dir="/kaggle/input/ids-intrusion-csv",
    file_range=(6, 7),
    mask=10,
    label_col="Label",
    numeric_threshold=0.95
)

combined_test = combine_to_df_for_range(
    input_dir="/kaggle/input/ids-intrusion-csv",
    file_range=(4, 5),
    mask=10,
    label_col="Label",
    numeric_threshold=0.95
)

combined_val = combine_to_df_for_range(
    input_dir="/kaggle/input/ids-intrusion-csv",
    file_range=(7, 8),
    mask=10,
    label_col="Label",
    numeric_threshold=0.95
)

combined_train.to_csv("/kaggle/working/combined_train.csv", index=False)
combined_supplement.to_csv("/kaggle/working/combined_supplement.csv", index=False)
combined_test.to_csv("/kaggle/working/combined_test.csv", index=False)
combined_val.to_csv("/kaggle/working/combined_val.csv", index=False)